In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add a and b"""
    return int(a) + int(b)

@tool
def subtract(a: int, b: int) -> int:
    """Subtract b from a"""
    return int(a) - int(b)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b"""
    return int(a) * int(b)

print(f"Tool Name: {multiply.name}")
print(f"Description: {multiply.description}")
print(f"Args: {multiply.args}")

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tool Name: multiply
Description: Multiply a and b
Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [5]:
# Custom tool with Pydantic schema
from pydantic import BaseModel, Field

class Search_Input(BaseModel):
    query: str = Field(description="should be search query")

@tool("Search-Tool", args_schema=Search_Input)
def search(query: str) -> str:
    """Look up things online."""
    return "LangChain"


In [6]:
# Creating tool without decorator
def mymultiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return int(a) * int(b)

class MultiplyInput(BaseModel):
    """Multiply two integers together."""
    a: int = Field(..., description="First integer")
    b: int = Field(..., description="Second integer")

from langchain_core.tools import StructuredTool

mytool = StructuredTool.from_function(
    func=mymultiply,
    description="Multiply 2 numbers",
    args_schema=MultiplyInput,
    name="sivamultiply"
)

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavilysearch = TavilySearchResults(max_results=5)

search_results = tavilysearch.invoke("Tell me about SivaPrasad Valluru")
print(f"Search Results: {search_results}")

C:\Users\Aditya\AppData\Local\Temp\ipykernel_12152\1181370508.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavilysearch = TavilySearchResults(max_results=5)


Search Results: [{'title': 'SivaPrasad Valluru is a instructor with 20 years of ... - Instagram', 'url': 'https://www.instagram.com/p/CrYfQAZtUzh/', 'content': 'SivaPrasad Valluru is a instructor with 20 years of experince in IT . He has worked with Motorola, Alcatel Lucent and TechMahindra earlier. Now, he delivers corporate trainings.  \n  \nHe has done contribution to Spring framework. He got certified as a instructor from Mulesoft and Pivotal and has delivered many certified trainings. [...] He spends a lot of time in understanding the frameworks and explores the internal workings. Since he knows most of the internals, he delivers training with confidence. [...] While delivering trainings, he never uses high level jargons and presentations. He goes to low level and makes the participants visualize everything. He feels that Learning Why, When and Where to use a product makes you a better developer. Once your core concepts are clear, other advanced topics will become easy to understa

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

tools = [tavilysearch, add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke([HumanMessage(content="Hi!")])
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

response = llm_with_tools.invoke([HumanMessage(content="Tell me about ShivPrasad Valluru")])
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")


ContentString: Hello! I'm a large language model, able to perform a wide variety of tasks. What can I do for you today?
ToolCalls: []
ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': 'ShivPrasad Valluru'}, 'id': '2593d5b3-6357-4345-8232-b33c6120ba36', 'type': 'tool_call'}]


In [10]:
from typing import List
from langchain_core.tools import BaseTool

def find_tool_by_name(tools: List[BaseTool], tool_name: str) -> BaseTool:
    for tool in tools:
        if tool.name == tool_name:
            return tool
    raise ValueError(f"Tool with name {tool_name} not found")

In [17]:
# ============================================
# OPTION 1: Using Modern AgentExecutor (RECOMMENDED)
# ============================================
print("\n" + "="*60)
print("OPTION 1: Using Modern AgentExecutor (RECOMMENDED)")
print("="*60)


OPTION 1: Using Modern AgentExecutor (RECOMMENDED)


In [19]:
from langchain.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate
from langchain_core.tools.render import render_text_description
from langchain.agents.output_parsers.react_single_input import ReActSingleInputOutputParser
from langchain.agents.format_scratchpad.log import format_log_to_str
from langchain_core.runnables import RunnablePassthrough

# Create ReAct prompt template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

# Create prompt
prompt = PromptTemplate.from_template(template=react_template).partial(
    tools=render_text_description(tools),
    tool_names=", ".join([t.name for t in tools]),
)

# Create LLM
llm_agent = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0,
    stop=["\nObservation"]
)

# Create agent runnable
agent_runnable = (
    RunnablePassthrough.assign(
        agent_scratchpad=lambda x: format_log_to_str(x["intermediate_steps"]),
    )
    | prompt
    | llm_agent
    | ReActSingleInputOutputParser()
)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

# Execute queries
print("\nQuery 1: What is 2+3 multiplied by 4")
try:
    result = agent_executor.invoke({"input": "What is 2+3 multiplied by 4"})
    print(f"\nFinal Answer: {result['output']}\n")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

print("\nQuery 2: Who is Sivaprasad Valluru")
try:
    result = agent_executor.invoke({"input": "Who is Sivaprasad Valluru"})
    print(f"\nFinal Answer: {result['output']}\n")
except Exception as e:
    print(f"Error: {e}")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_core.prompts import PromptTemplate

# Create LLM for agent
llm_agent = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0
)

# Use initialize_agent which works with all LangChain versions
agent_executor = initialize_agent(
    tools=tools,
    llm=llm_agent,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    early_stopping_method="generate"
)

# Execute queries
print("\nQuery 1: What is 2+3 multiplied by 4")
try:
    result = agent_executor.invoke({"input": "What is 2+3 multiplied by 4"})
    print(f"Final Answer: {result['output']}\n")
except Exception as e:
    print(f"Error: {e}")

print("\nQuery 2: Who is Sivaprasad Valluru")
try:
    result = agent_executor.invoke({"input": "Who is Sivaprasad Valluru"})
    print(f"Final Answer: {result['output']}\n")
except Exception as e:
    print(f"Error: {e}")

ImportError: cannot import name 'initialize_agent' from 'langchain.agents' (d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
# OPTION 2: Manual Implementation with Modern Imports
# ============================================
print("\n" + "="*60)
print("OPTION 2: Manual Implementation (For Learning)")
print("="*60)

from langchain.agents.output_parsers.react_single_input import ReActSingleInputOutputParser
from langchain.agents.format_scratchpad.log import format_log_to_str
from langchain_core.agents import AgentAction, AgentFinish
from typing import Union

# Create prompt
prompt_manual = PromptTemplate.from_template(template=react_template).partial(
    tool_names=", ".join([t.name for t in tools]),
    tools=render_text_description(tools)
)

# Create LLM with stop sequences
llm_manual = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0,
    stop=["\nObservation"]
)

# Create agent chain
agent_manual = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_log_to_str(x["intermediate_steps"]),
    }
    | prompt_manual
    | llm_manual
    | ReActSingleInputOutputParser()
)

# Manual agent loop function
def run_manual_agent(query: str, max_iterations: int = 10):
    """Run manual ReAct agent loop"""
    intermediate_steps = []
    iterations = 0
    
    while iterations < max_iterations:
        iterations += 1
        print(f"\n--- Iteration {iterations} ---")
        
        try:
            agent_step: Union[AgentAction, AgentFinish] = agent_manual.invoke({
                "input": query,
                "intermediate_steps": intermediate_steps,
            })
            
            print(f"Agent Step: {agent_step}")
            
            # Check if finished
            if isinstance(agent_step, AgentFinish):
                print(f"\nFinal Answer: {agent_step.return_values}")
                return agent_step.return_values
            
            # Execute action
            if isinstance(agent_step, AgentAction):
                tool_name = agent_step.tool
                tool_to_use = find_tool_by_name(tools, tool_name)
                tool_input = agent_step.tool_input
                
                # Handle tool input parsing
                if isinstance(tool_input, str) and "," in tool_input:
                    # Try to parse comma-separated values
                    tool_input_list = [x.strip() for x in tool_input.split(",")]
                    if len(tool_input_list) > 1:
                        tool_input = {
                            "a": tool_input_list[0],
                            "b": tool_input_list[1]
                        }
                
                print(f"Tool Input: {tool_input}")
                
                try:
                    observation = tool_to_use.invoke(tool_input)
                except Exception as e:
                    observation = f"Error: {str(e)}"
                
                print(f"Observation: {observation}")
                
                intermediate_steps.append((agent_step, str(observation)))
                print(f"Intermediate Steps Count: {len(intermediate_steps)}")
                
        except Exception as e:
            print(f"Error in iteration: {str(e)}")
            return {"output": f"Error: {str(e)}"}
    
    return {"output": "Max iterations reached"}

# Run manual agent
print("\nManual Agent - Query 1: What is 2+3 multiplied by 4")
result = run_manual_agent("What is 2+3 multiplied by 4")

print("\n\nManual Agent - Query 2: Who is Sivaprasad Valluru")
result = run_manual_agent("Who is Sivaprasad Valluru")

In [ ]:
# ============================================
# OPTION 3: Step-by-step Manual Execution (Original Approach Fixed)
# ============================================
print("\n" + "="*60)
print("OPTION 3: Step-by-step Manual Execution")
print("="*60)

intermediate_steps = []

# First invocation
print("\n--- Step 1 ---")
agentaction = agent_manual.invoke({
    "input": "What is 2+3 multiplied by 4",
    "intermediate_steps": intermediate_steps,
})
print(f"Action: {agentaction}")

if isinstance(agentaction, AgentAction):
    # Simulate observation
    intermediate_steps.append((agentaction, "5"))
    print(f"Added to intermediate_steps: {intermediate_steps}")

# Second invocation
print("\n--- Step 2 ---")
agentaction = agent_manual.invoke({
    "input": "What is 2+3 multiplied by 4",
    "intermediate_steps": intermediate_steps,
})
print(f"Action: {agentaction}")

if isinstance(agentaction, AgentAction):
    intermediate_steps.append((agentaction, "20"))
    print(f"Added to intermediate_steps: {intermediate_steps}")

# Third invocation
print("\n--- Step 3 ---")
agentaction = agent_manual.invoke({
    "input": "What is 2+3 multiplied by 4",
    "intermediate_steps": intermediate_steps,
})
print(f"Action: {agentaction}")

if isinstance(agentaction, AgentFinish):
    print(f"Final Answer: {agentaction.return_values}")

print("\n" + "="*60)
print("Formatted Scratchpad:")
print(format_log_to_str(intermediate_steps))
print("="*60)